# Sesión 01 (adaptado) — Ciclo de Vida de ML con League of Legends
**Curso:** Ingeniería del Conocimiento (ISO56B)
**Basado en:** Sesión 01 — Ciclo de Vida de Proyectos de Machine Learning (Dataset Heart Disease)
**Dataset nuevo:** League of Legends Diamond Ranked Games (10 min)

---

## Objetivo
Replicar el ciclo de vida completo de un proyecto de Machine Learning (recolección, exploración,
preparación, entrenamiento, evaluación y despliegue conceptual) usando un dataset distinto:
partidas de League of Legends en rango Diamante, con el objetivo de predecir si el
**equipo azul gana la partida** a partir de las estadísticas de los primeros 10 minutos de juego.


## 1. Configuración del entorno
Importamos las mismas librerías que en la Sesión 01. En Google Colab la mayoría ya están disponibles.

In [ ]:
# Librerías principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay
)
import joblib

# Configuración visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print('Entorno configurado correctamente.')


---
## 2. Etapa 1 — Recolección de datos

Dataset: **League of Legends Diamond Ranked Games (10 min)**
Fuente: Kaggle — https://www.kaggle.com/datasets/bobbyscience/league-of-legends-diamond-ranked-games-10-min

Cada fila es una partida de rango Diamante-Maestro. Las columnas describen el estado de la
partida a los **10 minutos** de juego (kills, oro, experiencia, torres, dragones, wards, etc.)
para ambos equipos.

| Variable (ejemplos) | Descripción |
|---|---|
| `blueWins` | **Target**: 1 = ganó el equipo azul, 0 = ganó el equipo rojo |
| `blueKills` / `redKills` | Asesinatos de cada equipo a los 10 min |
| `blueDeaths` / `redDeaths` | Muertes de cada equipo |
| `blueTotalGold` / `redTotalGold` | Oro total acumulado |
| `blueGoldDiff` | Diferencia de oro (azul - rojo) |
| `blueExperienceDiff` | Diferencia de experiencia (azul - rojo) |
| `blueDragons` / `redDragons` | Dragones asesinados |
| `blueTowersDestroyed` / `redTowersDestroyed` | Torres destruidas |
| `blueWardsPlaced` / `redWardsPlaced` | Wards (visión) colocados |

**Antes de ejecutar esta celda:** sube el archivo `high_diamond_ranked_10min.csv`
(descargado de Kaggle) a la sesión de Colab (ícono de carpeta → *Subir archivo*),
o descomenta el bloque de `files.upload()`.

In [ ]:
# 2.1 Carga del dataset
# Si estás en Colab y aún no subiste el archivo, descomenta estas líneas:
# from google.colab import files
# uploaded = files.upload()

CSV_PATH = 'high_diamond_ranked_10min.csv'

try:
    df = pd.read_csv(CSV_PATH)
    print(f'Dataset cargado — {df.shape[0]} registros, {df.shape[1]} columnas')
except FileNotFoundError:
    print(f'No se encontró {CSV_PATH}. Sube el archivo a la sesión de Colab e inténtalo de nuevo.')
    raise

df.head()


---
## 3. Etapa 2 — Exploración y preparación de datos

In [ ]:
# 3.1  Información general del dataset
df.info()
df.describe().round(2)


In [ ]:
# 3.2  Verificación de valores nulos y duplicados
print(df.isnull().sum())
print(f'Registros duplicados: {df.duplicated().sum()}')
df = df.drop_duplicates().reset_index(drop=True)


In [ ]:
# 3.3  Distribución de la variable objetivo (blueWins)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

conteo = df['blueWins'].value_counts()
axes[0].bar(['Gana Rojo (0)', 'Gana Azul (1)'], conteo.reindex([0, 1]))
axes[0].set_title('Conteo de partidas por resultado')
axes[0].set_ylabel('Número de partidas')

axes[1].pie(conteo.reindex([0, 1]), labels=['Gana Rojo', 'Gana Azul'],
            autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporción de victorias')

plt.tight_layout()
plt.show()


In [ ]:
# 3.4  Distribución de variables numéricas clave
num_cols = ['blueGoldDiff', 'blueExperienceDiff', 'blueKills', 'blueDeaths', 'blueDragons']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.histplot(data=df, x=col, hue='blueWins', kde=True, ax=axes[i], bins=25)
    axes[i].set_title(col)
axes[-1].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# 3.5  Matriz de correlación
corr = df.drop(columns=['gameId']).corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(16, 14))
sns.heatmap(corr, mask=mask, annot=False, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Matriz de correlación — League of Legends (10 min)')
plt.tight_layout()
plt.show()

# Variables más correlacionadas con el target
print(corr['blueWins'].sort_values(ascending=False))


---
## 4. Etapa 3 — Preparación de datos para el modelado

In [ ]:
# 4.1  Separación de features y target
# 'gameId' es solo un identificador de partida, no aporta información predictiva
X = df.drop(columns=['gameId', 'blueWins'])
y = df['blueWins']

feature_names = X.columns.tolist()

print(f'Features: {X.shape}')
print(f'Target:   {y.shape}')


In [ ]:
# 4.2  División train / test (80-20, estratificado)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} partidas | Test: {X_test.shape[0]} partidas')
print(f'Proporción de victorias azules — train: {y_train.mean():.3f} | test: {y_test.mean():.3f}')


---
## 5. Etapa 4 — Entrenamiento de modelos
Entrenamos los mismos tres modelos base mediante **pipelines** de Scikit-learn.

In [ ]:
# 5.1  Definición de pipelines
pipelines = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'DecisionTree': Pipeline([
        ('clf', DecisionTreeClassifier(max_depth=5, random_state=42))
    ]),
    'RandomForest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=7, random_state=42))
    ])
}

# 5.2  Entrenamiento y validación cruzada (5-fold)
results = {}
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    results[name] = scores
    print(f'{name:25s}  Accuracy CV: {scores.mean():.4f} ± {scores.std():.4f}')


In [ ]:
# 5.3  Comparación visual de modelos
fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(results.values(), tick_labels=results.keys())
ax.set_ylabel('Accuracy (5-fold CV)')
ax.set_title('Comparación de modelos — Validación cruzada')
plt.tight_layout()
plt.show()


---
## 6. Etapa 5 — Evaluación del mejor modelo

In [ ]:
# 6.1  Seleccionar el mejor modelo según CV y entrenar con todo el train set
best_name = max(results, key=lambda k: results[k].mean())
best_pipeline = pipelines[best_name]
best_pipeline.fit(X_train, y_train)

y_pred = best_pipeline.predict(X_test)

print(f'Mejor modelo: {best_name}')
print(f'Accuracy en test: {best_pipeline.score(X_test, y_test):.4f}')
print('\n' + '='*60)
print('CLASSIFICATION REPORT')
print('='*60)
print(classification_report(y_test, y_pred, target_names=['Gana Rojo', 'Gana Azul']))


In [ ]:
# 6.2  Matriz de confusión
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Gana Rojo', 'Gana Azul'],
    cmap='Blues', ax=ax
)
ax.set_title(f'Matriz de Confusión — {best_name}')
plt.tight_layout()
plt.show()


In [ ]:
# 6.3  Curva ROC (solo si el modelo soporta predict_proba)
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_estimator(best_pipeline, X_test, y_test, ax=ax, name=best_name)
ax.plot([0, 1], [0, 1], 'k--', label='Aleatorio')
ax.set_title(f'Curva ROC — {best_name}')
ax.legend()
plt.tight_layout()
plt.show()


---
## 7. Etapa 6 — Serialización y despliegue conceptual
Guardamos el pipeline entrenado con `joblib` para simular la etapa de despliegue.

In [ ]:
# 7.1  Guardar el modelo
MODEL_PATH = 'lol_win_prediction_model.joblib'
joblib.dump(best_pipeline, MODEL_PATH)
print(f'Modelo guardado en: {MODEL_PATH}')


In [ ]:
# 7.2  Cargar y verificar el modelo serializado
loaded_model = joblib.load(MODEL_PATH)

assert np.array_equal(loaded_model.predict(X_test), y_pred), 'Error: predicciones no coinciden'
print('Verificación OK: el modelo cargado reproduce las predicciones.')


In [ ]:
# 7.3  Función de inferencia (simula un endpoint de predicción)

def predict_blue_win(match_stats: dict) -> dict:
    """
    Recibe un diccionario con las estadísticas de una partida a los 10 minutos
    y retorna la predicción (gana azul / gana rojo) y la probabilidad.
    'match_stats' debe tener las mismas claves que 'feature_names'.
    """
    model = joblib.load(MODEL_PATH)

    try:
        ordered_values = [match_stats[name] for name in feature_names]
        input_df = pd.DataFrame([ordered_values], columns=feature_names)
    except KeyError as e:
        raise ValueError(f'Falta la estadística: {e}. Revisa feature_names para la lista completa.')

    pred = model.predict(input_df)[0]
    proba = model.predict_proba(input_df)[0]

    prediction_label = 'Gana equipo Azul' if pred == 1 else 'Gana equipo Rojo'

    return {
        'prediccion': prediction_label,
        'probabilidad_gana_azul': round(float(proba[1]), 4),
        'probabilidad_gana_rojo': round(float(proba[0]), 4)
    }

# Ejemplo de uso: partida donde el equipo azul va ganando en oro y kills a los 10 min
partida_ejemplo = X_test.iloc[0].to_dict()

resultado = predict_blue_win(partida_ejemplo)
print('\n--- Resultado de inferencia ---')
for k, v in resultado.items():
    print(f'  {k}: {v}')

print(f'\n(Resultado real de esta partida: {"Gana Azul" if y_test.iloc[0] == 1 else "Gana Rojo"})')


---
## 8. Resumen del ciclo de vida recorrido

| Etapa | Qué hicimos | Herramientas |
|---|---|---|
| **Recolección** | Descarga del dataset LoL Diamond Ranked (Kaggle) | `pandas` |
| **Exploración** | Análisis de distribuciones, correlaciones, nulos | `seaborn`, `matplotlib` |
| **Preparación** | Limpieza, split estratificado 80/20 | `train_test_split` |
| **Entrenamiento** | 3 modelos con pipelines + validación cruzada 5-fold | `Pipeline`, `cross_val_score` |
| **Evaluación** | Accuracy, precision, recall, F1, ROC, matriz confusión | `classification_report`, `RocCurveDisplay` |
| **Despliegue** | Serialización con joblib + función de inferencia | `joblib` |


---
## 9. Ejercicios propuestos (adaptados)

### Ejercicio 1 — Experimentación con hiperparámetros
Utilice `GridSearchCV` o `RandomizedSearchCV` para optimizar los hiperparámetros del
`RandomForestClassifier` sobre este dataset. Compare el accuracy antes y después del tuning.

### Ejercicio 2 — Importancia de variables
Usando `best_pipeline.named_steps['clf'].feature_importances_` (si el mejor modelo es un árbol
o RandomForest), identifique las 5 variables más importantes para predecir la victoria.
¿Tiene sentido desde la lógica del juego (oro, experiencia, dragones)?

### Ejercicio 3 — Registro de experimentos
Investigue la librería **MLflow** (`!pip install mlflow`) e implemente un registro básico de
las métricas y parámetros de cada modelo entrenado en este notebook.

---
*Notebook adaptado — Ciclo de Vida de ML aplicado a League of Legends*